# Flood segmentation: graph cut vs U-Net vs SegFormer-B0

**Task.** Binary semantic segmentation of flooded areas (flood = water, flooded road, flooded building) in UAV imagery.

**Research question.** Are hand-designed colour and region cues enough to map floodwater, or does it take learned features and global context? How much does a from-scratch model suffer on limited data compared with a pretrained one?

| Slot | Method | Code |
|---|---|---|
| A. Classical | Graph cut: colour/texture GMM unaries, contrast-sensitive Potts, our own Boykov-Kolmogorov max-flow | `src/methods/classical.py` |
| B. From scratch | U-Net written in PyTorch, random init | `src/methods/scratch.py` |
| C. Pretrained | SegFormer-B0 (MiT-B0 ImageNet encoder), fine-tuned | `src/methods/pretrained.py` |

All three share the same FloodNet splits (`splits/`), the same metrics (`src/eval/metrics.py`), and the same `load`/`predict` interface.

## 0. Setup
In Colab this clones the repo, installs requirements, and keeps checkpoints on Google Drive. Set `TRAIN = True` to retrain the deep models; otherwise existing checkpoints are reused.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
TRAIN = False
TRAIN_ABLATION = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/ccs-250-final-project"):
        !git clone -q https://github.com/athenavillarin/ccs-250-final-project.git /content/ccs-250-final-project
    %cd /content/ccs-250-final-project
    !pip install -q -r requirements.txt
    os.environ["FLOOD_DATA_DIR"] = "/content/data"
    os.environ["FLOOD_CHECKPOINT_DIR"] = "/content/drive/MyDrive/flood/checkpoints"
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import torch
from IPython.display import Image as Show, display

from src.utils.paths import CHECKPOINT_DIR, DATA_DIR, RESULTS_DIR
from src.utils.seed import seed_everything

seed_everything()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1. Data
FloodNet Supervised v1.0 (official train/val/test) and the Kaggle Flood Area Segmentation set for out-of-domain testing. Images are stored at 1024 px on the long side. Regenerating the splits must leave `splits/` unchanged.

In [ ]:
if not (DATA_DIR / "floodnet" / "test").exists() or not (DATA_DIR / "kaggle").exists():
    !python scripts/download_data.py
!python scripts/make_splits.py
!git status --short splits

In [ ]:
from src.data.dataset import load_pair, read_split
from src.utils.viz import overlay, plt

stats = []
for split in ("train", "val", "test", "kaggle"):
    ids = read_split(split)
    flood = np.mean([load_pair(split, i)[1].mean() for i in ids[:: max(1, len(ids) // 100)]])
    stats.append({"split": split, "images": len(ids), "flood_pixels_%": round(100 * flood, 1)})
display(pd.DataFrame(stats))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (split, i) in zip(axes, [("train", 0), ("train", 50), ("test", 7), ("kaggle", 3)]):
    image, mask = load_pair(split, read_split(split)[i])
    ax.imshow(overlay(image, mask))
    ax.set_title(f"{split}: flood mask in blue")
    ax.axis("off")
plt.show()

## 2. Classical: graph cut
### 2.1 Max-flow correctness
Our Boykov-Kolmogorov solver is checked on random grids against the max-flow value from SciPy (used only as a test oracle) and against the max-flow/min-cut theorem.

In [ ]:
!python -m pytest -q tests

### 2.2 Synthetic sanity check
A winding river and a pond with known ground truth, rendered with heavy Gaussian noise. Pixel-wise labeling (lambda = 0) is compared with the graph cut.

In [ ]:
!python scripts/sanity_classical.py
display(Show(str(RESULTS_DIR / "sanity_classical.png")), Show(str(RESULTS_DIR / "sanity_classical_sweep.png")))

### 2.3 Fit on train, tune lambda, sigma, and bias on val
Flood and background GMMs over Lab colour plus local lightness std are fit on the train split; lambda, sigma, and the flood bias are chosen on a fixed val subset only. The graph cut runs at 256 px and the mask is upsampled for evaluation.

In [ ]:
if TRAIN or not (CHECKPOINT_DIR / "classical.npz").exists():
    !python scripts/fit_classical.py
display(pd.read_csv(RESULTS_DIR / "classical_tuning.csv").sort_values("miou", ascending=False).head(5))
display(Show(str(RESULTS_DIR / "classical_tuning.png")))

## 3. From scratch: U-Net
Four-level U-Net (batch norm, transposed-conv upsampling, bottleneck dropout), BCE + Dice loss, flips / random resized crops / colour jitter, AdamW with weight decay, cosine schedule, early stopping on val mIoU.

In [ ]:
from src.eval.benchmark import count_params
from src.methods.scratch import UNet

print(f"U-Net parameters: {count_params(UNet()):.2f} M")
if TRAIN or not (CHECKPOINT_DIR / "scratch_train_s42.pt").exists():
    !python scripts/train.py --method scratch --split train --device {DEVICE}
display(Show(str(RESULTS_DIR / "curves_scratch_train_s42.png")))

## 4. Pretrained: SegFormer-B0
ImageNet-pretrained MiT-B0 encoder with a one-logit decode head, normalised with the statistics of its own image processor; logits are upsampled to the input size before thresholding. Same splits, loss, and augmentation as the U-Net.

In [ ]:
from src.methods.pretrained import SegFormer

print(f"SegFormer-B0 parameters: {count_params(SegFormer()):.2f} M")
if TRAIN or not (CHECKPOINT_DIR / "pretrained_train_s42.pt").exists():
    !python scripts/train.py --method pretrained --split train --device {DEVICE}
display(Show(str(RESULTS_DIR / "curves_pretrained_train_s42.png")))

### 4.1 Training on 25% and 50% of the data (for the ablation)

In [ ]:
for method in ("scratch", "pretrained"):
    for split in ("train_50", "train_25"):
        if TRAIN_ABLATION or not (CHECKPOINT_DIR / f"{method}_{split}_s42.pt").exists():
            !python scripts/train.py --method {method} --split {split} --device {DEVICE}
display(pd.read_csv(RESULTS_DIR / "training_runs.csv"))

## 5. Experiments
Predictions are cached under `data/preds/`, so this cell is fast after the first run.

In [ ]:
!python scripts/run_experiments.py --device {DEVICE}

### 5.1 Experiment 1: main comparison on the FloodNet test split

In [ ]:
display(pd.read_csv(RESULTS_DIR / 'main_results.csv'))

### 5.2 Experiment 2: data-size ablation

In [ ]:
display(pd.read_csv(RESULTS_DIR / "ablation_data_size.csv"))
display(Show(str(RESULTS_DIR / "ablation_data_size.png")))

### 5.3 Experiment 3: efficiency (CPU vs GPU runtime, parameters, training time)

In [ ]:
display(pd.read_csv(RESULTS_DIR / "efficiency.csv"))
display(Show(str(RESULTS_DIR / "efficiency_runtime.png")))

### 5.4 Experiment 4: out-of-domain test (Kaggle Flood Area Segmentation)

In [ ]:
display(pd.read_csv(RESULTS_DIR / "ood_kaggle.csv"))
display(Show(str(RESULTS_DIR / "ood_comparison.png")))

### 5.5 Experiment 5: failure analysis
Share of each original FloodNet class predicted as flood (high is good for flood classes, low for the others), then the hardest and easiest flooded test images.

In [ ]:
per_class = pd.read_csv(RESULTS_DIR / "failure_per_class.csv")
display(per_class.pivot(index="class", columns="method", values="pred_flood_pct").round(1))
display(Show(str(RESULTS_DIR / "failure_per_class.png")))
display(Show(str(RESULTS_DIR / "failure_cases.png")), Show(str(RESULTS_DIR / "qualitative_examples.png")))

## 6. Demo: all three methods on one new image
The same `load`/`predict` calls the FastAPI backend uses.

In [ ]:
import time

from src.methods import METHODS

image, gt = load_pair("kaggle", read_split("kaggle")[12])
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
axes[0].imshow(overlay(image, gt))
axes[0].set_title("Ground truth")
for ax, (key, info) in zip(axes[1:], METHODS.items()):
    model = info["module"].load(None, "cpu" if key == "classical" else DEVICE)
    start = time.perf_counter()
    mask = info["module"].predict(model, image)
    ax.imshow(overlay(image, mask))
    ax.set_title(f"{info['name']}: {100 * mask.mean():.0f}% flooded, {1000 * (time.perf_counter() - start):.0f} ms")
for ax in axes:
    ax.axis("off")
plt.show()